# 295. Find Median from Data Stream

**Difficulty:** Hard &nbsp;|&nbsp; **Topics:** design, heap, two-pointers, sorting
&nbsp;|&nbsp; [LeetCode](https://leetcode.com/problems/find-median-from-data-stream/)

The **median** is the middle value in an ordered integer list. If the size of the
list is even, there is no middle value, and the median is the mean of the two middle
values.

- For `arr = [2,3,4]`, the median is `3`.
- For `arr = [2,3]`, the median is `(2 + 3) / 2 = 2.5`.

Implement the `MedianFinder` class:

- `MedianFinder()` initialises the object.
- `addNum(num)` adds the integer `num` from the data stream to the data structure.
- `findMedian()` returns the median of all elements so far. Answers within `10^-5`
  of the actual answer will be accepted.

---

### Example

```
MedianFinder m = new MedianFinder();
m.addNum(1);        // arr = [1]
m.addNum(2);        // arr = [1, 2]
m.findMedian();     // 1.5   = (1 + 2) / 2
m.addNum(3);        // arr = [1, 2, 3]
m.findMedian();     // 2.0
```

---

### Constraints

- `-10^5 <= num <= 10^5`
- There will be at least one element in the data structure before calling `findMedian`
- At most `5 * 10^4` calls will be made to `addNum` and `findMedian`

**Follow-ups:**
1. If all integers from the stream are in the range `[0, 100]`, how would you optimise?
2. If **99%** of them are in `[0, 100]`, how would you optimise?

The first Hard in the repo, and it earns it by making you give something up. You
learned in #1396 that a running sum cannot give you back the median - here is the
problem that forces you to build the thing that can.

## Before you write anything

**1.** The obvious answer: keep a list, `bisect.insort` each new number, and index the
middle. `insort` finds the position in `O(log n)` and then **shifts** every later element
one place - so what is the real cost per `addNum`? What is `findMedian`? Work out the
total for `5 * 10^4` calls, then be fair to it: the shift is a C `memmove`, so measure it
before you dismiss it. (You will, at the bottom of this notebook.)

**2.** Here is the idea the whole problem turns on. To know the median you do **not**
need the sorted order - you need the **one or two elements in the middle**. So split the
numbers into two halves: a **low** half and a **high** half, of equal size (or with low
one bigger). Then:

```
odd  count -> the median is the LARGEST  of the low half
even count -> the median is the mean of the largest of low and the smallest of high
```

Write down the single operation each half must support in `O(1)`, and name the structure
that does it.

**3.** A heap gives you the min (or the max) of a collection in `O(1)`, with `O(log n)`
insert and removal. Python's `heapq` is a **min-heap only** - there is no max-heap. So
how do you make one? (One character per push and per pop. Write out what
`heappush(low, -num)` means when you read it back.)

**4.** **The balance invariant**, and this is the part to get exactly right. Write it as
a formula relating `len(low)` and `len(high)`. Then say what goes wrong if you allow the
two sizes to drift by two instead of one - be specific about which element you would
return and why it is not the middle.

**5.** The push looks strange the first time. You do **not** compare `num` to anything to
decide which heap it belongs in. Instead:

```
push num onto low, then move low's largest over to high, then if high is bigger, move its
smallest back
```

Three heap operations for one number. Work through why this always puts `num` in the
right half without a single `if num < ...` comparison, and why the naive
"compare and choose" version needs a rebalance anyway. Fewer branches, and no case to
forget.

**6.** `findMedian` must return a **float**, and the even case is a mean of two numbers.
In Python `/` already gives a float, but check: what does your odd case return, and is
`3` acceptable where `3.0` is expected? (Here, yes - but say why, and say when it would
not be.)

**7.** Follow-up 1: every number is in `[0, 100]`. Suddenly you do not need heaps at all -
101 counters will do. What is `addNum` then, and what is `findMedian`? Which is `O(1)`
and which is `O(101)`, and why is `O(101)` still `O(1)`?

**8.** Follow-up 2: **99%** are in `[0, 100]` and 1% are anywhere. Why does the counting
answer break, and what is the smallest change that fixes it? (You do not need a new idea -
you need two of the ideas you already have, side by side.)

## Two routes

**A - two heaps** *(write this first)*

```
self.low  = []      # a MAX-heap, by negation: the smaller half
self.high = []      # a MIN-heap:              the larger half
```

Invariant: `len(low) == len(high)` or `len(low) == len(high) + 1`. So `low` never has
fewer, and never more than one extra.

`addNum` is the three-step dance from question 5. `findMedian` is: if the halves are
equal in size, average `-low[0]` and `high[0]`; otherwise return `-low[0]`.

Costs: `addNum` is `O(log n)`, `findMedian` is `O(1)`, memory `O(n)`. At `5 * 10^4`
numbers that is about 800 000 comparisons total - against roughly 600 million element
moves for the insort version, most of them very fast. Measure both.

**B - counting buckets** *(follow-up 1)*

```
self.counts = [0] * 101
self.n = 0
```

`addNum` is one increment - genuinely `O(1)`, no heap, no log. `findMedian` walks the
101 buckets accumulating counts until it passes the middle, which is `O(101)` - a
constant, and a small one.

Then follow-up 2 in one sentence: keep the buckets for `[0, 100]` **and** two heaps (or
two sorted lists) for the outliers below and above, plus the counts of each. The median
walk starts by asking "how many are below 0?" and continues into the buckets. That is
the whole trick, and it is what real percentile systems do - a fast path for the common
range, an exact path for the tail.

> **You cannot get the median back from a sum.** #1396 kept `(total, count)` and could
> never answer "what was the typical journey?" - only "what was the mean?", which is not
> the same question and is much easier to distort. This problem is what it costs to
> answer it properly: `O(log n)` per insert and `O(n)` memory, and it is the reason
> every monitoring dashboard you have ever seen shows p50 and p99 rather than an average.

In [ ]:
class MedianFinder:

    def __init__(self):
        pass

    def addNum(self, num: int) -> None:
        pass

    def findMedian(self) -> float:
        pass

### The test harness

`findMedian` is a pure query - unlike #146's `get` and #362's `getHits`, asking does not
change anything. So this harness can do what those two could not: probe after **every
single** `addNum`.

`check` replays numbers against your class and against the obvious model - keep them all,
sort, take the middle - and after every `addNum` compares `findMedian()` to the model, to
within `10^-5`. A number pushed into the wrong half is therefore reported at the `addNum`
that caused it, not fifty calls later.

It also checks that the answer is a **number** and not, say, an integer index or a `None`
from a method that forgot to return, and it prints the two middle values of the model on
failure so you can see which side your answer fell on.

`stress` covers the shapes that break a balance invariant: already-sorted input,
reverse-sorted input, all-identical values, and heavy duplicates - each of which pushes
every number to the same side. Run this cell; don't edit it.

In [ ]:
import random


def true_median(sorted_vals):
    n = len(sorted_vals)
    mid = n // 2
    if n % 2:
        return float(sorted_vals[mid])
    return (sorted_vals[mid - 1] + sorted_vals[mid]) / 2


def check(nums, name=""):
    '''Add each number, and after EVERY add compare findMedian() to a sort-the-lot model.'''
    log = []
    try:
        mf = MedianFinder()
    except Exception as e:
        return False, [f"   !! MedianFinder() raised {type(e).__name__}: {e}"]

    seen = []
    for i, x in enumerate(nums):
        try:
            mf.addNum(x)
        except Exception as e:
            log.append(f"   !! addNum({x}) raised {type(e).__name__}: {e}")
            return False, log

        seen.append(x)
        seen.sort()
        want = true_median(seen)

        try:
            got = mf.findMedian()
        except Exception as e:
            log.append(f"   !! after addNum({x}), findMedian() raised {type(e).__name__}: {e}")
            return False, log

        if not isinstance(got, (int, float)) or isinstance(got, bool):
            log.append(f"   !! findMedian() must return a number, got {type(got).__name__}: {got!r}")
            return False, log
        if abs(float(got) - want) > 1e-5:
            mid = len(seen) // 2
            middle = seen[mid - 1:mid + 1] if len(seen) % 2 == 0 else [seen[mid]]
            log.append(f"   !! after add #{i + 1} (addNum({x})), findMedian() is {got!r},"
                       f" should be {want}")
            log.append(f"      {len(seen)} numbers; the middle {'two are' if len(middle) == 2 else 'one is'} {middle}")
            log.append(f"      sorted so far: {seen[:12]}{' ...' if len(seen) > 12 else ''}")
            return False, log

        log.append(f"addNum({x:>7}) -> findMedian() = {got}")

    return True, log


def stress(n, seed=0, lo=-1000, hi=1000):
    random.seed(seed)
    return check([random.randint(lo, hi) for _ in range(n)])


def report(name, ok, log, tail=5):
    print(f"{'OK  ' if ok else 'FAIL'} {name}")
    if not ok:
        for line in log[-tail:]:
            print(f"       {line}")

In [ ]:
# tests
CASES = [
    ("the LeetCode example",              [1, 2, 3]),
    ("a single number",                   [5]),
    ("two numbers - an even median",      [1, 2]),
    ("negatives",                         [-1, -2, -3, -4]),
    ("mixed signs straddling zero",       [-5, 5, -3, 3, 0]),
    ("already sorted - everything goes right",  list(range(1, 21))),
    ("reverse sorted - everything goes left",   list(range(20, 0, -1))),
    ("all identical",                     [7] * 15),
    ("heavy duplicates",                  [3, 1, 3, 1, 3, 1, 2, 2, 2]),
    ("a half-and-half median",            [1, 1, 1, 1, 100, 100, 100, 100]),
    ("the value ceilings",                [-10**5, 10**5, 0, -10**5, 10**5]),
    ("one huge outlier does not move the median much",
                                          [1, 2, 3, 4, 5, 10**5]),
    ("alternating extremes",              [x for pair in zip(range(1, 11), range(100, 90, -1))
                                           for x in pair]),
    ("a long ascending run then a long descending one",
                                          list(range(1, 51)) + list(range(50, 0, -1))),
]

for name, nums in CASES:
    report(name, *check(nums))

for n, seed, lo, hi in [(50, 1, -10, 10), (200, 2, -1000, 1000),
                        (1000, 3, 0, 3), (2000, 4, -10**5, 10**5)]:
    report(f"stress: {n} numbers (seed {seed}, range {lo}..{hi})", *stress(n, seed, lo, hi))

print("\ntrace of the LeetCode example:")
for line in check([1, 2, 3])[1]:
    print("  " + line)

## After it passes

- **Check your invariant continuously.** Add
  `assert len(self.low) - len(self.high) in (0, 1)` at the end of `addNum` and run
  everything again. If it never fires, you have a proof rather than a hope - and if it
  fires, it fires at the `addNum` that broke it rather than at the median that looked odd.
- **Race the naive version.** Write the `bisect.insort` one from question 1, run the same
  tests, then `timeit` both at 50 000 numbers. Predict the winner first. The insort
  version may well be *faster* at this size, because a `memmove` of 50 000 machine words
  beats 17 Python-level heap comparisons - and finding that out is more valuable than
  being told. Then push both to 10^6 and see where the curves cross.
- **Build route B** (101 buckets), run the same tests with values clamped to `[0, 100]`,
  and time it against the heaps. Then answer follow-up 2 by putting the two together and
  say, in one sentence, which numbers take the fast path.
- **Add `findPercentile(p)`** - p50 is the median, p99 is the number 99% of the data is
  below. Which of your two routes extends to it cheaply, and which one does not? That
  question is the entire reason monitoring systems store histograms rather than samples.
- **Then look back at #1396.** It kept `(total, count)` per route and could only ever
  report a mean. Add a `MedianFinder` per route to that class and it can report the
  *typical* journey time - which is what a passenger actually experiences, and what an
  average hides the moment one train breaks down. Cost it: what does that do to the
  memory claim you proved in that notebook?
- Siblings: #480 Sliding Window Median (this, plus expiry - genuinely hard),
  #703 Kth Largest Element in a Stream (one heap instead of two - do it first if the
  heaps did not click), **#1244 Design A Leaderboard** (the other "I only need part of the
  order" problem), #1396 Design Underground System (the mean this replaces).